In [0]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("PatientDataAnalysis").getOrCreate()


Tasks

1. Create DataFrame


In [0]:
df = spark.createDataFrame(data,columns)


2. Add required derived columns


In [0]:
from pyspark.sql.functions import col
df_with_derived = df.withColumn("total_bill", col("consultation_fee") + (col("tests_count") * 500))

df_with_derived.show()

+--------+------------+---------+-----------+----------------+-----------+----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_bill|
+--------+------------+---------+-----------+----------------+-----------+----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5500|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      4000|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2000|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      6000|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7500|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      4500|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|      5500|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|      2500|
+--------+------------+---------+-----------+---------

3. Filter high-value patients


In [0]:
df_with_derived.filter(col("total_bill") > 5000).show()

+--------+------------+---------+----------+----------------+-----------+----------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|total_bill|
+--------+------------+---------+----------+----------------+-----------+----------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|          1|      5500|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|      6000|
|     105|Vikram Singh|  Chennai| Neurology|            7000|          1|      7500|
|     107| Karan Patel|Ahmedabad|Cardiology|            5000|          1|      5500|
+--------+------------+---------+----------+----------------+-----------+----------+



4. Perform aggregation by department


In [0]:
from pyspark.sql.functions import count, avg, sum

agg_df = df_with_derived.groupBy("department").agg(
    count("visit_id").alias("patient_count"),
    avg("consultation_fee").alias("avg_fee"),
    sum("total_bill").alias("total_revenue")
)
agg_df.show()

+-----------+-------------+-------+-------------+
| department|patient_count|avg_fee|total_revenue|
+-----------+-------------+-------+-------------+
| Cardiology|            3| 5000.0|        17000|
|Orthopedics|            2| 3000.0|         8500|
|Dermatology|            2| 1500.0|         4500|
|  Neurology|            1| 7000.0|         7500|
+-----------+-------------+-------+-------------+



5. Sort results appropriately

In [0]:

sorted_agg_df = agg_df.orderBy(col("total_revenue").desc())
sorted_agg_df.show()

+-----------+-------------+-------+-------------+
| department|patient_count|avg_fee|total_revenue|
+-----------+-------------+-------+-------------+
| Cardiology|            3| 5000.0|        17000|
|Orthopedics|            2| 3000.0|         8500|
|  Neurology|            1| 7000.0|         7500|
|Dermatology|            2| 1500.0|         4500|
+-----------+-------------+-------+-------------+



🔷 PART 2 — SPARK SQL
Tasks
1. Convert DataFrame to a temp view


In [0]:
df_with_derived.createOrReplaceTempView("hospital_records")

2. Write SQL to:\
Fetch specific department records\
Calculate revenue per city\
Identify top patients\
Count patients per department

In [0]:
spark.sql("""
SELECT * FROM hospital_records
where department = 'Cardiology'
""").show()

+--------+------------+---------+----------+----------------+-----------+----------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|total_bill|
+--------+------------+---------+----------+----------------+-----------+----------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|          1|      5500|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|      6000|
|     107| Karan Patel|Ahmedabad|Cardiology|            5000|          1|      5500|
+--------+------------+---------+----------+----------------+-----------+----------+



In [0]:
city_revenue = spark.sql("""
    SELECT city, SUM(total_bill) as city_revenue
    FROM hospital_records
    GROUP BY city
    ORDER BY city_revenue DESC
""")
city_revenue.show()

+---------+------------+
|     city|city_revenue|
+---------+------------+
|Bangalore|        8500|
|  Chennai|        7500|
|Hyderabad|        5500|
|Ahmedabad|        5500|
|  Kolkata|        4500|
|    Delhi|        4000|
|   Mumbai|        2000|
+---------+------------+



In [0]:
top_patients = spark.sql("""
    SELECT patient_name, city, department, total_bill
    FROM hospital_records
    ORDER BY total_bill DESC
    LIMIT 3
""")
top_patients.show()

+------------+---------+----------+----------+
|patient_name|     city|department|total_bill|
+------------+---------+----------+----------+
|Vikram Singh|  Chennai| Neurology|      7500|
|  Priya Nair|Bangalore|Cardiology|      6000|
| Arjun Reddy|Hyderabad|Cardiology|      5500|
+------------+---------+----------+----------+



In [0]:
dept_counts = spark.sql("""
    SELECT department, COUNT(visit_id) as patient_count
    FROM hospital_records
    GROUP BY department
    ORDER BY patient_count DESC
""")
dept_counts.show()

+-----------+-------------+
| department|patient_count|
+-----------+-------------+
| Cardiology|            3|
|Orthopedics|            2|
|Dermatology|            2|
|  Neurology|            1|
+-----------+-------------+



🔷 PART 3 — DELTA LAKE (CORE
OPERATIONS)
Tasks
1. Create a Delta table from the dataset
2. Insert new records
3. Update existing records
4. Delete specific records
5. Perform an UPSERT using MERGE

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("new_df")



In [0]:
spark.sql("""
    INSERT INTO new_df VALUES 
    (109, 'Zoya Khan', 'Pune', 'Neurology', 7000, 2),
    (110, 'Rohan Das', 'Mumbai', 'Cardiology', 5000, 0),
    (111, 'Rohan Das', 'Mumbai', 'Radiology', 5000, 0)

""")



DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("UPDATE new_df SET consultation_fee = consultation_fee * 1.1 WHERE department = 'Cardiology'")

DataFrame[num_affected_rows: bigint]

In [0]:
spark.sql("DELETE FROM new_df WHERE department = 'Cardiology'")

DataFrame[num_affected_rows: bigint]

In [0]:
df.createOrReplaceTempView("source_updates")
spark.sql("""
    MERGE INTO new_df AS target
    USING source_updates AS source
    ON target.visit_id = source.visit_id
    WHEN MATCHED THEN
      UPDATE SET *
    WHEN NOT MATCHED THEN
      INSERT *
""")
spark.sql("SELECT * FROM new_df WHERE visit_id IN (108, 111)").show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     111|   Rohan Das|   Mumbai|  Radiology|            5000|          0|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



🔷 PART 4 — DELTA ADVANCED
Tasks
1. Retrieve table history


In [0]:
spark.sql("DESCRIBE HISTORY new_df").show()

+-------+-------------------+---------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|         userId|            userName|           operation| operationParameters| job|          notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+---------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|     26|2026-05-04 12:38:28|140745818471898|nfazuser5819_mml....|            OPTIMIZE|{clusterBy -> [],...|NULL|{4181754920009164}|   0ebef80c-9e4f-4a0...|0504-120126-p

2. Query an older version of the table


In [0]:
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table("new_df")
df_v0.show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



3. Explain the effect of VACUUM


4. Execute VACUUM (dry run)

🔷 PART 5 — PARQUET → DELTA
Tasks
1. Save dataset as Parquet


In [0]:
sample = spark.createDataFrame(data,columns)
sample.write.format("parquet").mode("overwrite").saveAsTable("hospital_parquet_table")


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7626822303560823>, line 2
      1 sample = spark.createDataFrame(data,columns)
----> 2 sample.write.format("parquet").mode("overwrite").saveAsTable("hospital_parquet_table")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id

2. Convert Parquet dataset to Delta


In [0]:
parquet_df = spark.table("hospital_parquet_table")

parquet_df.write.format("delta").mode("overwrite").saveAsTable("hospital_delta_table")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5882443243960710>, line 3
      1 parquet_df = spark.table("hospital_parquet_table")
----> 3 parquet_df.write.format("delta").mode("overwrite").saveAsTable("hospital_delta_table")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.u

3. Validate conversion

In [0]:
table_info = spark.sql("DESCRIBE DETAIL hospital_delta_table")
table_info.select("name", "format", "location").show()

spark.sql("DESCRIBE HISTORY hospital_delta_table").select("version", "operation").show()

🔷 PART 6 — INCREMENTAL LOAD
Scenario
You receive daily updates.
Tasks
1. Create a target Delta table


In [0]:
data = [
    (101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
    (102,"Sneha Kapoor","Delhi","Orthopedics",3000,2)
]
columns = ["visit_id", "patient_name", "city", "department", "consultation_fee", "tests_count"]
spark.createDataFrame(data, columns).write.format("delta").mode("overwrite").saveAsTable("patient_master")

2. Create a dataset representing daily updates


In [0]:
daily_updates = [
    (101, "Arjun Reddy", "Hyderabad", "Cardiology", 5000, 3), 
    (112, "Suresh Raina", "Chennai", "Neurology", 7000, 1)     
]

updates_df = spark.createDataFrame(daily_updates, columns)
updates_df.createOrReplaceTempView("daily_batch_view")

3. Implement incremental load logic


In [0]:
%sql
MERGE INTO patient_master AS target
USING daily_batch_view AS source
ON target.visit_id = source.visit_id
WHEN MATCHED THEN
  UPDATE SET 
    target.tests_count = source.tests_count
WHEN NOT MATCHED THEN
  INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0


4. Ensure:
existing records are updated
new records are inserted

In [0]:
from delta.tables import DeltaTable

master_table = DeltaTable.forName(spark, "patient_master")

master_table.alias("target") \
    .merge(updates_df.alias("source"), "target.visit_id = source.visit_id") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()\
    .show()

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|                2|               1|               0|                1|
+-----------------+----------------+----------------+-----------------+



🔷 PART 7 — DLT PIPELINE
Tasks
1. Create Bronze table (inline data)


2. Create Silver table with transformations


3. Create Gold table with aggregations


4. Define proper dependencies


5. Run the pipeline

PART 8 — UNITY CATALOG
Tasks
1. Create a catalog


In [0]:
%sql
-- SQL
CREATE CATALOG IF NOT EXISTS hospital_catalog_1;
USE CATALOG hospital_catalog_1;


2. Create a schema


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hospital_schema;
USE SCHEMA hospital_schema;



3. Create a table inside schema


In [0]:
%sql
CREATE TABLE IF NOT EXISTS hospital_schema.hospital_visits (
  visit_id LONG,
  patient_name STRING,
  city STRING,
  department STRING,
  consultation_fee LONG,
  tests_count LONG
);

4. Load data into the table

In [0]:
%sql
INSERT INTO TABLE hospital_schema.hospital_visits
VALUES
  (1, 'John Doe', 'New York', 'Cardiology', 500, 2),
  (2, 'Jane Smith', 'Los Angeles', 'Orthopedics', 700, 3)

num_affected_rows,num_inserted_rows
2,2


PART 9 — DATA GOVERNANCE
Tasks
1. Discover tables using catalog


In [0]:
%sql
SHOW TABLES IN hospital_catalog_1.hospital_schema;

database,tableName,isTemporary
hospital_schema,hospital_visits,false
,daily_batch_view,true
,hospital_records,true
,source_updates,true


2. Create a derived table



In [0]:
%sql
CREATE TABLE main.default.patient_summary
USING DELTA
AS SELECT patient_id, count(*) as visit_count
FROM main.default.patient_parquet_table
GROUP BY patient_id;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8466995061927851>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE TABLE main.default.patient_summary\nUSING DELTA\nAS SELECT patient_id, count(*) as visit_count\nFROM main.default.hospital_visits\nGROUP BY patient_id;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_ma

3. Observe lineage


4. Apply access control

5. Query audit logs

🔥 FINAL CAPSTONE\
Build a system that:\
Raw Data → Clean Data → Analytics → Governed Data\
Requirements\
Use Delta tables\
Use incremental logic\
Build DLT pipeline\
Create a Simple Unity Catalog structure

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS hospital_logistics;
CREATE SCHEMA IF NOT EXISTS hospital_logistics.patient_ops;

USE CATALOG hospital_logistics;
USE SCHEMA patient_ops;

In [0]:
import dlt
from pyspark.sql.functions import col, lit, current_timestamp, sum

# 1. BRONZE: Raw Data Ingestion
@dlt.table(name="raw_patient_data")
def raw_patient_data():
    # In production, use spark.readStream for true incremental ingestion
    return spark.read.format("delta").table("source_system_table")

# 2. SILVER: Incremental Cleaning & Logic
@dlt.table(name="cleaned_patient_data")
@dlt.expect_or_fail("valid_id", "visit_id IS NOT NULL")
def cleaned_patient_data():
    return dlt.read("raw_patient_data") \
        .withColumn("total_bill", col("consultation_fee") + (col("tests_count") * 500)) \
        .withColumn("last_updated", current_timestamp())

# 3. GOLD: Business Analytics
@dlt.table(name="revenue_summary")
def revenue_summary():
    return dlt.read("cleaned_patient_data") \
        .groupBy("department", "city") \
        .agg(sum("total_bill").alias("total_revenue"))

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-4604808684138908>, line 1
----> 1 import dlt
      2 from pyspark.sql.functions import col, lit, current_timestamp, sum
      4 # 1. BRONZE: Raw Data Ingestion

ModuleNotFoundError: No module named 'dlt'